In [2]:
import functools
from typing import List

import numpy as np
import jax
import jax.numpy as jnp

from jax.experimental.pallas.ops.tpu.megablox.gmm import gmm
from jax.experimental.shard_map import shard_map
from jax.sharding import Mesh, NamedSharding, PartitionSpec

/home/lsiyuan_google_com/miniconda3/envs/torch312/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
P = PartitionSpec

mesh = Mesh(jax.devices(), ('x',))

seq_p_spec = P('x',)
seq_p_sharding = NamedSharding(mesh, seq_p_spec)

In [4]:
def print_sharded(x):
    if not isinstance(x, jax.Array):
        print("not a JAX arry")
        return
    print(f"x shape: {x.shape} x sharding: {x.sharding}")
    if not x.is_fully_replicated:
        for i, shard in enumerate(x.addressable_shards):
            print(f"shard {i}: {shard.data}")
    else:
        print(f"x is fully replicated: \n {x}")

In [5]:
N_TOKENS = 128 # number of tokens in the current batch
n_tokens = N_TOKENS
E = 16 # number of global experts
EP_SIZE = mesh.size
n_shards = EP_SIZE
D = 256 # n_head * head_dim
F = 512 # intermediate size
TOP_K = 2
dtype = jnp.bfloat16

In [6]:
x = jax.random.normal(jax.random.PRNGKey(0), (N_TOKENS, D))
expert_gating_weight = jax.random.normal(jax.random.PRNGKey(1), (D, E))
gmm_weight = jax.random.normal(jax.random.PRNGKey(2), (E, D, 2 * F))
gmm_weight2 = jax.random.normal(jax.random.PRNGKey(3), (E, F, D))

In [7]:
def moe_naive_impl(x, expert_gating_weight, gmm_weight, gmm_weight2, top_k):
    n_tokens = x.shape[0]
    
    gating_out = x @ expert_gating_weight
    gating_logits = jax.nn.softmax(gating_out, axis=-1)
    # top_k_indices: [T, TOP_K]
    top_k_logits, top_k_indices = jax.lax.top_k(gating_logits, top_k)

    token_outputs = []

    for i in range(n_tokens):
        curr_token = jnp.expand_dims(x[i], axis=0) # [1, D]
        assigned_expert_idxs = top_k_indices[i] # [TOP_K] idx for the top_k selected experts for token i
        tok_expert_act = []
        for expert_idx in assigned_expert_idxs:
            # act for expert_idx's expert
            expert_weight_1 = gmm_weight[expert_idx] # [D, 2 * F]
            expert_weight_2 = gmm_weight2[expert_idx] # [F, D]

            gmm_1_out = curr_token @ expert_weight_1 # [1, 2 * F]
            # print(gmm_1_out.shape)
            gmm1_w1_proj, gmm1_w3_proj = jnp.split(gmm_1_out, 2, axis=-1)
            act = jax.nn.silu(gmm1_w1_proj * gmm1_w3_proj) # [1, F]
            # print(act.shape)
            gmm_2_out = act @ expert_weight_2 # [1, D]
            tok_expert_act.append(gmm_2_out)
        
        # Now tok_expert_act has act for all of the selected experts, each one is [1, D]
        # concat them. [top_k, D]
        experts_act = jnp.concatenate(tok_expert_act, axis=0) 

        # Take weighted sum of acts using topk logits
        # Note: weighted sum might be done after gmm_1, different from model to model
        top_k_weights = top_k_logits[i] 
        top_k_weights = jnp.expand_dims(top_k_weights, axis=1) # [top_k, 1]
        act = jnp.sum(experts_act * top_k_weights, axis=0, keepdims=True)

        token_outputs.append(act)
    
    return jnp.concatenate(token_outputs, axis=0)

out = moe_naive_impl(x, expert_gating_weight, gmm_weight, gmm_weight2, TOP_K)
print(out.shape)

(128, 256)


In [8]:
# From https://github.com/pytorch/xla/blob/2c343188f33c71939a3e20cd504746ef70244211/torch_xla/experimental/custom_kernel.py#L1387
def _round_up_to_multiple_of_128_within_limit(x: int, limit: int) -> int:
    """
    Rounds the given integer `x` up to the nearest multiple of 128, without exceeding
    the specified `limit`.

    If `x` is less than or equal to 128, returns 128.
    If `x` is less than `limit`, returns the smallest multiple of 128 greater than or
    equal to `x`.
    If `x` is greater than or equal to `limit`, searches for the largest multiple of
    128 less than or equal to `limit` (down to 512) that divides `x` evenly, and
    returns it.
    If no such candidate is found, returns `limit`.

    Args:
        x (int): The integer to round up.
        limit (int): The upper bound (must be a multiple of 128 and at least 128).

    Returns:
        int: The rounded value according to the rules above.

    Raises:
        AssertionError: If `limit` is less than 128 or not a multiple of 128.
    """
    assert limit >= 128 and limit % 128 == 0
    if x <= 128:
        return 128
    if x < limit:
        return (x + 127) // 128 * 128
    for candidate in range(limit, 511, -128):
        if x % candidate == 0:
            return candidate
    return limit

# From https://github.com/pytorch/xla/blob/2c343188f33c71939a3e20cd504746ef70244211/torch_xla/experimental/custom_kernel.py#L1387
def _get_tiling_size_for_gmm_kernel(m: int, k: int, n: int,
                                    g: int) -> tuple[int, int, int]:
    """
    Calculate optimal tiling sizes for a GMM kernel in a Mixture of Experts
    (MoE) setting.

    Args:
        m (int): The total number of tokens.
        n (int): The output feature dimension.
        k (int): The input feature dimension.
        g (int): The number of experts.

    Returns:
        tuple[int, int, int]: A tuple (tm, tk, tn)
    """

    # TODO(Chengji): increase the upper limit tiling size of m when we can set
    # the vmem size to be used for gmm kernel.
    # NOTE: In average each expert has m // g tokens, but as it might be unbalanced,
    # here we doubled the token size when choosing tiling size of m. 2m//g can be
    # either greater or less than 512. If there are 32 tokens and topk=2,
    # m=topk * num_tokens=64, in this case, 2*m//g will be less than 512.
    tm = _round_up_to_multiple_of_128_within_limit(2 * m // g, 512)
    tm = min(tm, m)  # there's a requirement that m % tm == 0
    # k/n correspond to n_input_features/n_output_features in the matmul so they are
    # normally greater than 2048, unless the num shards is large.
    tk = _round_up_to_multiple_of_128_within_limit(k, 2048)
    tn = _round_up_to_multiple_of_128_within_limit(n, 2048)
    return tm, tk, tn


def _gmm(lhs, rhs, group_sizes, group_offset=None):
    # Current logic to determine tile size in torch_xla.
    # If don't use this tiling calculation, gmm would error out with the
    # default tiling.
    m, k, g = lhs.shape[0], lhs.shape[1], rhs.shape[0]
    n = rhs.shape[1]
    tm, tk, tn = _get_tiling_size_for_gmm_kernel(m, k, n, g)
    
    return gmm(lhs, rhs,
               group_sizes,
               group_offset=group_offset,
               tiling=(tm, tk, tn),
               preferred_element_type=jnp.bfloat16)

In [9]:
expert_gating_weight = jax.random.normal(jax.random.PRNGKey(1), (D, E))

expert_gating_weight = jax.device_put(expert_gating_weight, NamedSharding(mesh, P()))

In [10]:
functools.partial(jax.jit,
                  static_argnames=["n_experts",
                                   "top_k"])
def expert_gating(x,
                  expert_gating_weight,
                  n_experts,
                  top_k):
    @functools.partial(shard_map,
                       mesh=mesh,
                       in_specs=(x_sharding,
                                 P()),
                       out_specs=(x_sharding, P('x',), P('x',), P('x',)),
                       check_rep=False)
    def expert_gating_internal(x,
                               expert_gating_weight):
        gating_out = x @ expert_gating_weight
        gating_logits = jax.nn.softmax(gating_out, axis=-1)
        top_k_logits, top_k_indices = jax.lax.top_k(gating_logits, top_k)
        top_k_indices = top_k_indices.flatten()
        top_k_logits = top_k_logits.flatten()
    
        topk_argsort_indices = jnp.argsort(top_k_indices)
        topk_argsort_revert_indices = topk_argsort_indices.argsort()

        n_tokens = x.shape[0]
        token_indices = jnp.arange(n_tokens).repeat(top_k)
        sorted_token_indices = token_indices[topk_argsort_indices]
    
        histo = jnp.bincount(top_k_indices, length=n_experts)

        sorted_tokens = x[sorted_token_indices]

        return sorted_tokens, histo, top_k_logits, topk_argsort_revert_indices

    return expert_gating_internal(x, expert_gating_weight)

In [11]:
x_sharding = P('x', None)
x = jax.device_put(x, NamedSharding(mesh, x_sharding))

sorted_tokens, histo, top_k_logits, topk_argsort_revert_indices = expert_gating(x, expert_gating_weight, n_experts=E, top_k=TOP_K)

In [12]:
@functools.partial(shard_map,
                       mesh=mesh,
                       in_specs=(P('x'),
                                 P('x'),
                                 P()),
                       out_specs=(P('x')),
                       check_rep=False)
def calc_ep_group_metadata(histo, ep_ranks, ep_group_mask):
    ep_group_size = E // EP_SIZE
    ep_idx = ep_ranks[0]
    
    ep_group_send_size = histo @ ep_group_mask
    return ep_group_send_size

# Should generate offline
def gen_ep_group_size_mask(n_experts, ep_size):
    mask = jnp.zeros((ep_size, n_experts), dtype=jnp.int32)
    ep_group_size = n_experts // ep_size
    for i in range(ep_size):
        mask = mask.at[i, i*ep_group_size : (i+1)*ep_group_size].set(1)
    return mask.transpose()

In [13]:
mask = gen_ep_group_size_mask(E, EP_SIZE)
ep_ranks = jnp.arange(EP_SIZE)
ep_ranks = jax.device_put(ep_ranks, NamedSharding(mesh, P('x')))

ep_group_sizes = calc_ep_group_metadata(histo, ep_ranks, mask)

In [14]:
@functools.partial(shard_map,
                   mesh=mesh,
                   in_specs=(P('x'), P('x')),
                   out_specs=(P('x'), P('x'), P('x'), P('x'), P('x'), P('x')),
                   check_rep=False)
def compute_ra2a_metadata2(send_sizes: jax.Array, # int32[EP_SIZE], indicating the number of tokens to send to each EP rank
                           ep_rank: jax.Array): # arange(ep_size) used for indexing inside shard_mapd
    all_send_sizes = jax.lax.all_gather(send_sizes, 'x')
    row_cum_sum = jnp.cumsum(all_send_sizes, axis=1)
    col_cum_sum = jnp.cumsum(all_send_sizes, axis=0)

    def _append_zero_and_remove_last(x, append_row):
        '''
        Append an all-zero row/col and remove the last row/col from the
        original input.

        Args:
        x (jax.Array): 2D Array
        append_row (bool): if append along row
        '''
        if append_row:
            zero_row = jnp.zeros((1, x.shape[1]), dtype=x.dtype)
            res = jnp.concatenate([zero_row, x[:-1]], axis=0)
        else:
            zero_col = jnp.zeros((x.shape[0], 1), dtype=x.dtype)
            res = jnp.concatenate([zero_col, x[:, :-1]], axis=1)
        return res

    full_input_offsets = _append_zero_and_remove_last(row_cum_sum, False)
    full_ouptut_offsets = _append_zero_and_remove_last(col_cum_sum, True)
    all_recv_sizes = all_send_sizes.transpose()

    input_offsets = full_input_offsets[ep_rank[0]]
    ouptut_offsets = full_ouptut_offsets[ep_rank[0]]
    recv_sizes = all_recv_sizes[ep_rank[0]]
    
    full_combine_input_offsets = full_ouptut_offsets.transpose()
    full_combine_output_offsets = full_input_offsets.transpose()

    combine_send_sizes = all_recv_sizes[ep_rank[0]]

    combine_input_offsets = full_combine_input_offsets[ep_rank[0]]
    combine_output_offsets = full_combine_output_offsets[ep_rank[0]]
    
    return input_offsets, ouptut_offsets, recv_sizes, combine_send_sizes, combine_input_offsets, combine_output_offsets

shmapped_ra2a = shard_map(functools.partial(jax.lax.ragged_all_to_all, axis_name='x'),
                          mesh=mesh,
                          in_specs=(seq_p_spec,
                                    seq_p_spec,
                                    seq_p_spec,
                                    seq_p_spec,
                                    seq_p_spec,
                                    seq_p_spec),
                          out_specs=seq_p_spec,
                          check_rep=False)

In [16]:
output_buffer_shape = (sorted_tokens.shape[0] * n_shards, *sorted_tokens.shape[1:])
output_buffer = jnp.zeros(output_buffer_shape).astype(sorted_tokens.dtype)
output_buffer = jax.device_put(output_buffer, seq_p_sharding)

# Compute dispatch a2a metadata
input_offsets, output_offsets, recv_sizes, combine_send_sizes, combine_input_offsets, combine_output_offsets = compute_ra2a_metadata2(ep_group_sizes, ep_ranks)
combine_recv_sizes = ep_group_sizes

dispatch_output = shmapped_ra2a(sorted_tokens, output_buffer, input_offsets, ep_group_sizes, output_offsets, recv_sizes)

In [17]:
dispatch_output.shape

(2048, 256)

In [18]:
@functools.partial(shard_map,
                   mesh=mesh,
                   in_specs=(P('x', None),
                             P('x', None, None),
                             P('x'),
                             P('x', None, None)),
                   out_specs=P('x', None),
                   check_rep=False)
def moe_ffn(x, gmm1_weight, group_sizes, gmm2_weight):
    # gmm1_out = gmm(x, [w1, w2])
    # act = silu(out[:, :F] * out[:, F:])
    # out = gmm(act, w3)
    gmm1_out =  _gmm(x, gmm1_weight, group_sizes)
    gmm1_w1_proj, gmm1_w3_proj = jnp.split(gmm1_out, 2, axis=-1)
    act = jax.nn.silu(gmm1_w1_proj * gmm1_w3_proj)
    gmm2_out =  _gmm(act, gmm2_weight, group_sizes)
    return gmm2_out

In [19]:
group_sizes = combine_input_offsets
gmm_weight = jax.device_put(gmm_weight, NamedSharding(mesh, P('x', None, None)))
gmm_weight2 = jax.device_put(gmm_weight2, NamedSharding(mesh, P('x', None, None)))
                            
moe_ffn_out = moe_ffn(dispatch_output, gmm_weight, group_sizes, gmm_weight2)

In [20]:
moe_ffn_out.shape

(2048, 256)

In [21]:
combine_output_buffer = jnp.zeros_like(sorted_tokens, dtype=jnp.bfloat16)
combine_output = shmapped_ra2a(moe_ffn_out,
                               combine_output_buffer,
                               combine_input_offsets,
                               combine_send_sizes,
                               combine_output_offsets,
                               combine_recv_sizes)

In [22]:
output_orig_order = combine_output[topk_argsort_revert_indices]

In [23]:
weighted_out = jnp.sum((output_orig_order * jnp.expand_dims(top_k_logits, 1)).reshape(N_TOKENS, TOP_K, -1), axis=1)

In [24]:
weighted_out.shape

(128, 256)